# 01 · Data prep: remove duplicates, make leak-free splits

**Dataset:** [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) (4 classes: glioma, meningioma, notumor, pituitary).

**Why this notebook exists:** this dataset contains exact and near-duplicate images (the same scan re-saved with other contrast or sharpening), and many `Testing/` images are copies of `Training/` images. If we train on an image and then "test" on its copy, the accuracy is inflated. This notebook:

1. hashes every image (MD5 for exact copies, perceptual hash for near-copies),
2. groups copies together,
3. drops groups whose copies have **different labels**,
4. keeps **one image per group**,
5. splits the clean groups into **train / val / test (70 / 15 / 15)**, stratified by class. The original `Training/` / `Testing/` folders are not reused, because the test folder is full of training copies (in `notumor` almost all of it),
6. writes `splits.csv` + `data_report.json` for the training notebooks.

**Run on Kaggle:** *Add Input → Datasets →* `masoudnickparvar/brain-tumor-mri-dataset`. CPU is enough, no GPU needed.

In [ ]:
!pip install -q imagehash

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split


def find_data_dir() -> Path:
    if "NEUROLENS_DATA" in os.environ:
        return Path(os.environ["NEUROLENS_DATA"])
    # Kaggle mounts datasets under /kaggle/input (the exact sub-path has changed over time)
    for p in Path("/kaggle/input").glob("**/Training"):
        return p.parent
    raise FileNotFoundError("Dataset not found. Attach masoudnickparvar/brain-tumor-mri-dataset.")


DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
NEAR_DUP_BITS = 4      # pHash Hamming distance (out of 64) that counts as the same image
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
SEED = 42

## 1. Index all images

In [ ]:
rows = []
for source_split in ["Training", "Testing"]:
    for label in CLASSES:
        for p in sorted((DATA_DIR / source_split / label).glob("*")):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                rows.append({"path": str(p), "label": label, "source_split": source_split})

df = pd.DataFrame(rows)
assert len(df), f"No images found under {DATA_DIR}. Is the dataset attached?"
print(f"{len(df)} images")
df.groupby(["source_split", "label"]).size().unstack()

## 2. Hash every image

In [ ]:
def describe(path):
    data = Path(path).read_bytes()
    with Image.open(path) as im:
        w, h = im.size
        ph = str(imagehash.phash(im.convert("L")))
    return hashlib.md5(data).hexdigest(), ph, w, h

df[["md5", "phash", "width", "height"]] = [describe(p) for p in df.path]
print("exact duplicate files:", int(df.md5.duplicated().sum()))

## 3. Group exact and near-duplicates (union-find on pHash distance)

In [ ]:
H = np.frombuffer(b"".join(bytes.fromhex(h) for h in df.phash), dtype=np.uint8).reshape(-1, 8)
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
parent = np.arange(len(df))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

for i in range(len(df) - 1):
    dist = POPCOUNT[H[i + 1:] ^ H[i]].sum(axis=1)
    for j in np.nonzero(dist <= NEAR_DUP_BITS)[0]:
        ra, rb = find(i), find(i + 1 + j)
        if ra != rb:
            parent[rb] = ra

df["group"] = [find(i) for i in range(len(df))]

g = df.groupby("group")
sizes = g.size()
label_conflicts = g.label.nunique()
conflict_groups = label_conflicts[label_conflicts > 1].index
cross_split = g.source_split.nunique()
leak_groups = cross_split[cross_split > 1].index

print(f"unique images (groups):          {len(sizes)}")
print(f"images that have a copy:         {int(sizes[sizes > 1].sum())}")
print(f"groups with conflicting labels:  {len(conflict_groups)}")
print(f"groups in both Training/Testing: {len(leak_groups)}  <- train/test leakage")

### Look at the biggest duplicate groups

In [ ]:
import matplotlib.pyplot as plt

# Biggest groups first: each row should be the same scan with other contrast / sharpening
dup_groups = sizes[sizes > 1].sort_values(ascending=False).index[:4]
if len(dup_groups):
    fig, axes = plt.subplots(len(dup_groups), 5, figsize=(15, 3 * len(dup_groups)), squeeze=False)
    for r, grp in enumerate(dup_groups):
        members = df[df.group == grp].head(5)
        for c in range(5):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(members):
                m = members.iloc[c]
                ax.imshow(Image.open(m.path).convert("L"), cmap="gray")
                ax.set_title(f"{m.source_split}/{m.label} (group of {sizes[grp]})", fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "duplicate_examples.png", dpi=100)
    plt.show()

print("images kept per class (one per group):")
print(df.drop_duplicates("group").label.value_counts())

## 4. Clean and split

- Drop groups whose copies disagree on the label.
- Keep one image per group, so no scan appears twice anywhere.
- Split the clean images into train / val / test (70 / 15 / 15), stratified by class. Every copy of a scan was merged into one image above, so the three splits never share a scan.

In [ ]:
clean = df[~df.group.isin(conflict_groups)].copy()

# One image per group; prefer the Training/ copy so paths are stable between runs
clean = clean.sort_values(["group", "source_split", "path"], ascending=[True, False, True]).drop_duplicates("group")

trainval_idx, test_idx = train_test_split(
    clean.index, test_size=TEST_FRACTION, stratify=clean.label, random_state=SEED
)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=VAL_FRACTION / (1 - TEST_FRACTION), stratify=clean.loc[trainval_idx, "label"], random_state=SEED
)
clean.loc[train_idx, "split"] = "train"
clean.loc[val_idx, "split"] = "val"
clean.loc[test_idx, "split"] = "test"

summary = clean.groupby(["split", "label"]).size().unstack(fill_value=0).reindex(["train", "val", "test"])
summary["total"] = summary.sum(axis=1)
summary

## 5. Save

In [ ]:
out = clean.copy()
out["path"] = [Path(p).relative_to(DATA_DIR).as_posix() for p in out.path]  # portable across machines
out = out[["path", "label", "split", "source_split", "group", "md5", "phash", "width", "height"]]
out.to_csv(OUT_DIR / "splits.csv", index=False)

report = {
    "dataset": "masoudnickparvar/brain-tumor-mri-dataset",
    "near_dup_bits": NEAR_DUP_BITS,
    "images_total": int(len(df)),
    "exact_duplicate_files": int(df.md5.duplicated().sum()),
    "unique_groups": int(len(sizes)),
    "label_conflict_groups": int(len(conflict_groups)),
    "train_test_leak_groups": int(len(leak_groups)),
    "images_kept": int(len(out)),
    "split_counts": {k: int(v) for k, v in out.split.value_counts().items()},
    "split_by_class": {s: {k: int(v) for k, v in row.items()} for s, row in summary.drop(columns="total").iterrows()},
}
(OUT_DIR / "data_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

**Next:** download `splits.csv` and `data_report.json` from the Output tab, commit them to `ml/data/`, and fill the data table in the README. Then open `02_classifier.ipynb`.